In [6]:
# ── Setup for running Step 6 on Kaggle ───────────────────────────────────────
import pandas as pd
import numpy as np
import os
import time
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = '/kaggle/input/datasets/tracychanty/sentiment-extraction'
OUTPUT_DIR = '/kaggle/working/Outputs/sentiment_extraction'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MGMT_CHECKPOINT = f'{OUTPUT_DIR}/mgmt_scores_raw.csv'
QA_CHECKPOINT   = f'{OUTPUT_DIR}/qa_scores_raw.csv'
CHECKPOINT_EVERY = 500

# ── Load data ─────────────────────────────────────────────────────────────────
df         = pd.read_pickle(f'{DATA_DIR}/cleaned_transcripts_final.pkl')
df['transcript_clean'] = df['transcript_clean'].fillna('')
scores_raw = pd.read_csv(f'{DATA_DIR}/sentiment_scores_raw.csv')
print(f"df: {len(df)} rows")
print(f"scores_raw: {len(scores_raw)} rows")

# ── Load FinBERT ──────────────────────────────────────────────────────────────
MODEL_NAME = 'ProsusAI/finbert'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model      = model.to(device)
model.eval()
ID2LABEL   = {i: label.lower() for i, label in model.config.id2label.items()}
print(f"Device: {device}")
print(f"ID2LABEL: {ID2LABEL}")


# ── chunk_text function ───────────────────────────────────────────────────────
def chunk_text(text, max_tokens=512, overlap=50):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) == 0:
        return []
    effective_max = max_tokens - 2
    if len(tokens) <= effective_max:
        return [text]
    chunks = []
    start  = 0
    while start < len(tokens):
        end       = min(start + effective_max, len(tokens))
        chunk_ids = tokens[start:end]
        chunk_txt = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk_txt)
        if end == len(tokens):
            break
        start = end - overlap
    return chunks

# ── score_chunk function ──────────────────────────────────────────────────────
def score_chunk(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {'positive': 0.0, 'negative': 0.0, 'neutral': 1.0}
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=512, padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probs   = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
    prob_dict = {ID2LABEL[i]: float(probs[i]) for i in range(len(probs))}
    return {
        'positive': prob_dict.get('positive', 0.0),
        'negative': prob_dict.get('negative', 0.0),
        'neutral':  prob_dict.get('neutral',  0.0)
    }

# ── score_transcript function ─────────────────────────────────────────────────
def score_transcript(text, max_tokens=512, overlap=50):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {
            'positive_ratio': np.nan, 'negative_ratio': np.nan,
            'neutral_ratio':  np.nan, 'net_sentiment':  np.nan,
            'sentiment_volatility': np.nan, 'n_chunks': 0
        }
    chunks = chunk_text(text, max_tokens=max_tokens, overlap=overlap)
    if len(chunks) == 0:
        return {
            'positive_ratio': np.nan, 'negative_ratio': np.nan,
            'neutral_ratio':  np.nan, 'net_sentiment':  np.nan,
            'sentiment_volatility': np.nan, 'n_chunks': 0
        }
    chunk_scores = [score_chunk(c) for c in chunks]
    pos_scores   = [s['positive'] for s in chunk_scores]
    neg_scores   = [s['negative'] for s in chunk_scores]
    neu_scores   = [s['neutral']  for s in chunk_scores]
    net_scores   = [p - n for p, n in zip(pos_scores, neg_scores)]
    return {
        'positive_ratio':       float(np.mean(pos_scores)),
        'negative_ratio':       float(np.mean(neg_scores)),
        'neutral_ratio':        float(np.mean(neu_scores)),
        'net_sentiment':        float(np.mean(net_scores)),
        'sentiment_volatility': float(np.std(net_scores)) if len(net_scores) > 1 else 0.0,
        'n_chunks':             len(chunks)
    }

# ── Motley Fool parser ────────────────────────────────────────────────────────
def parse_motley_fool_transcript(transcript):
    if not isinstance(transcript, str) or len(transcript.strip()) == 0:
        return '', ''
    qa_markers = [
        'Questions and Answers', 'Question-and-Answer',
        'Q&A Session', 'QUESTION AND ANSWER', 'Questions & Answers'
    ]
    qa_start = None
    for marker in qa_markers:
        idx = transcript.find(marker)
        if idx != -1:
            qa_start = idx
            break
    if qa_start is not None:
        prepared_section = transcript[:qa_start]
        qa_section       = transcript[qa_start:]
    else:
        prepared_section = transcript
        qa_section       = ''
    speaker_pattern = re.compile(
        r'([A-Z][a-z]+(?:\s+[A-Z][a-z\-]+)+)\s+--\s+([^\n\.]{5,80}?)(?=\s+[A-Z]|\s*$)',
    )
    analyst_indicators = [
        'analyst', 'research', 'securities', 'capital', 'partners',
        'llc', 'inc', 'group', 'investments', 'bank', 'morgan',
        'goldman', 'jpmorgan', 'barclays', 'citi', 'ubs', 'wells fargo',
        'raymond james', 'cowen', 'piper', 'stifel', 'jefferies',
        'oppenheimer', 'needham', 'canaccord', 'macquarie'
    ]
    exec_indicators = [
        'chief', 'ceo', 'cfo', 'coo', 'cto', 'president', 'director',
        'officer', 'founder', 'chairman', 'vice president', 'vp',
        'head of', 'senior vice', 'executive vice', 'managing director',
        'investor relations', 'controller', 'treasurer', 'secretary'
    ]
    def classify_speaker(title):
        title_lower = title.lower()
        if 'operator' in title_lower:
            return 'operator'
        exec_score    = sum(1 for kw in exec_indicators    if kw in title_lower)
        analyst_score = sum(1 for kw in analyst_indicators if kw in title_lower)
        if exec_score > 0 and exec_score >= analyst_score:
            return 'exec'
        if analyst_score > 0:
            return 'analyst'
        return 'unknown'
    def extract_blocks(text, default_type):
        matches = list(speaker_pattern.finditer(text))
        if not matches:
            return [(default_type, text.strip())]
        blocks = []
        for i, match in enumerate(matches):
            title    = match.group(2).strip()
            spk_type = classify_speaker(title)
            if spk_type == 'unknown':
                spk_type = default_type
            speech_start = match.end()
            speech_end   = matches[i+1].start() if i+1 < len(matches) else len(text)
            speech       = text[speech_start:speech_end].strip()
            if speech and spk_type != 'operator':
                blocks.append((spk_type, speech))
        return blocks
    prepared_blocks = extract_blocks(prepared_section, 'exec')
    qa_blocks       = extract_blocks(qa_section,       'analyst')
    exec_parts      = [s for t, s in prepared_blocks if t == 'exec']
    exec_parts     += [s for t, s in qa_blocks       if t == 'exec']
    analyst_parts   = [s for t, s in qa_blocks       if t == 'analyst']
    return ' '.join(exec_parts).strip(), ' '.join(analyst_parts).strip()

# ── Apply parser to df ────────────────────────────────────────────────────────
print("Parsing speaker text...")
parsed             = df['transcript'].apply(parse_motley_fool_transcript)
df['exec_text']    = parsed.apply(lambda x: x[0])
df['analyst_text'] = parsed.apply(lambda x: x[1])
print(f"exec_text non-empty:    {(df['exec_text'].str.len() > 50).sum()}")
print(f"analyst_text non-empty: {(df['analyst_text'].str.len() > 50).sum()}")

# ── Rebuild df_merged from scores_raw ────────────────────────────────────────
print("\nRebuilding df_merged...")
df_reset  = df.reset_index().rename(columns={'index': 'original_idx'})
call_level = scores_raw.copy()
call_level['net_sentiment'] = call_level['positive_ratio'] - call_level['negative_ratio']

df_merged = df_reset.merge(
    call_level[[
        'original_idx', 'positive_ratio', 'negative_ratio',
        'neutral_ratio', 'net_sentiment', 'sentiment_volatility', 'n_chunks'
    ]],
    on='original_idx',
    how='inner'
).drop(columns=['original_idx'])

print(f"df_merged: {len(df_merged)} rows")
print(f"Columns:   {df_merged.columns.tolist()}")
print("\nAll setup complete — ready to run Step 6")

df: 14738 rows
scores_raw: 14738 rows


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda
ID2LABEL: {0: 'positive', 1: 'negative', 2: 'neutral'}
Parsing speaker text...
exec_text non-empty:    14712
analyst_text non-empty: 14379

Rebuilding df_merged...
df_merged: 14738 rows
Columns:   ['ticker', 'earnings_date', 'q', 'earnings_date_is_approx', 'transcript', 'transcript_clean', 'speaker_parse_success', 'return_5d', 'return_10d', 'return_20d', 'label_5d', 'label_10d', 'label_20d', 'exec_text', 'analyst_text', 'positive_ratio', 'negative_ratio', 'neutral_ratio', 'net_sentiment', 'sentiment_volatility', 'n_chunks']

All setup complete — ready to run Step 6


In [7]:
os.makedirs('/kaggle/working/Outputs/sentiment_extraction', exist_ok=True)

# ── Score management text (exec_text) ─────────────────────────────────────────
print("Scoring management remarks (exec_text)...")

mgmt_results      = []
processed_mgmt    = 0
start_time_mgmt   = time.time()

MGMT_CHECKPOINT  = '/kaggle/working/Outputs/sentiment_extraction/mgmt_scores_raw.csv'

# Load checkpoint if exists
if os.path.exists(MGMT_CHECKPOINT):
    mgmt_checkpoint_df  = pd.read_csv(MGMT_CHECKPOINT)
    mgmt_completed_idxs = set(mgmt_checkpoint_df['original_idx'].tolist())
    print(f"Resuming mgmt scoring from checkpoint: {len(mgmt_completed_idxs)} rows done")
else:
    mgmt_checkpoint_df  = pd.DataFrame()
    mgmt_completed_idxs = set()
    print("No mgmt checkpoint found — starting from scratch")

for idx, row in df_merged.iterrows():

    if idx in mgmt_completed_idxs:
        continue

    # Use exec_text if available, fall back to full transcript_clean
    text = row['exec_text'] if 'exec_text' in row.index else None
    if not isinstance(text, str) or len(text.strip()) == 0:
        text = row['transcript_clean'] if 'transcript_clean' in row.index else ''

    try:
        scores = score_transcript(text)
    except Exception as e:
        scores = {
            'positive_ratio':       np.nan,
            'negative_ratio':       np.nan,
            'neutral_ratio':        np.nan,
            'net_sentiment':        np.nan,
            'sentiment_volatility': np.nan,
            'n_chunks':             0
        }

    scores['original_idx']  = idx
    scores['ticker']        = row['ticker']
    scores['earnings_date'] = row['earnings_date']
    mgmt_results.append(scores)
    processed_mgmt += 1

    # Display progress statistics every 100 transcripts
    if processed_mgmt % 100 == 0 and processed_mgmt > 0:
        elapsed        = time.time() - start_time_mgmt
        rate           = processed_mgmt / elapsed
        remaining_rows = len(df_merged) - len(mgmt_completed_idxs) - processed_mgmt
        remaining      = max(0, remaining_rows) / rate / 60
        print(f"  Mgmt [{processed_mgmt}/{len(df_merged)}] | "
              f"Rate: {rate:.2f} rows/s | "
              f"ETA: {remaining:.1f}m")

    if len(mgmt_results) % CHECKPOINT_EVERY == 0:
        batch_df           = pd.DataFrame(mgmt_results)
        mgmt_checkpoint_df = pd.concat([mgmt_checkpoint_df, batch_df], ignore_index=True)
        mgmt_checkpoint_df.to_csv(MGMT_CHECKPOINT, index=False)
        mgmt_completed_idxs.update(batch_df['original_idx'].tolist())
        mgmt_results = []
        print(f"  ✓ Mgmt checkpoint saved: {len(mgmt_checkpoint_df)} rows total")

# Save remaining
if mgmt_results:
    batch_df           = pd.DataFrame(mgmt_results)
    mgmt_checkpoint_df = pd.concat([mgmt_checkpoint_df, batch_df], ignore_index=True)
    mgmt_checkpoint_df.to_csv(MGMT_CHECKPOINT, index=False)
    mgmt_completed_idxs.update(batch_df['original_idx'].tolist())

print(f"\nMgmt scoring complete: {len(mgmt_checkpoint_df)} rows")
print(f"Saved: {MGMT_CHECKPOINT}")

Token indices sequence length is longer than the specified maximum sequence length for this model (5656 > 512). Running this sequence through the model will result in indexing errors


Scoring management remarks (exec_text)...
No mgmt checkpoint found — starting from scratch
  Mgmt [100/14738] | Rate: 1.78 rows/s | ETA: 137.2m
  Mgmt [200/14738] | Rate: 1.73 rows/s | ETA: 140.3m
  Mgmt [300/14738] | Rate: 1.74 rows/s | ETA: 138.5m
  Mgmt [400/14738] | Rate: 1.73 rows/s | ETA: 138.2m
  Mgmt [500/14738] | Rate: 1.73 rows/s | ETA: 137.3m
  ✓ Mgmt checkpoint saved: 500 rows total
  Mgmt [600/14738] | Rate: 1.74 rows/s | ETA: 130.3m
  Mgmt [700/14738] | Rate: 1.77 rows/s | ETA: 127.3m
  Mgmt [800/14738] | Rate: 1.76 rows/s | ETA: 127.1m
  Mgmt [900/14738] | Rate: 1.76 rows/s | ETA: 126.4m
  Mgmt [1000/14738] | Rate: 1.77 rows/s | ETA: 124.8m
  ✓ Mgmt checkpoint saved: 1000 rows total
  Mgmt [1100/14738] | Rate: 1.77 rows/s | ETA: 119.1m
  Mgmt [1200/14738] | Rate: 1.77 rows/s | ETA: 118.2m
  Mgmt [1300/14738] | Rate: 1.77 rows/s | ETA: 116.9m
  Mgmt [1400/14738] | Rate: 1.77 rows/s | ETA: 116.3m
  Mgmt [1500/14738] | Rate: 1.77 rows/s | ETA: 115.0m
  ✓ Mgmt checkpoint sav

In [8]:
# ── Score Q&A text (analyst_text) ─────────────────────────────────────────────
print("Scoring Q&A section (analyst_text)...")

qa_results       = []
processed_qa     = 0
start_time_qa    = time.time()

QA_CHECKPOINT    = '/kaggle/working/Outputs/sentiment_extraction/qa_scores_raw.csv'

# Load checkpoint if exists
if os.path.exists(QA_CHECKPOINT):
    qa_checkpoint_df  = pd.read_csv(QA_CHECKPOINT)
    qa_completed_idxs = set(qa_checkpoint_df['original_idx'].tolist())
    print(f"Resuming Q&A scoring from checkpoint: {len(qa_completed_idxs)} rows done")
else:
    qa_checkpoint_df  = pd.DataFrame()
    qa_completed_idxs = set()
    print("No Q&A checkpoint found — starting from scratch")

for idx, row in df_merged.iterrows():

    if idx in qa_completed_idxs:
        continue

    text = row['analyst_text'] if 'analyst_text' in row.index else None

    if not isinstance(text, str) or len(text.strip()) == 0:
        qa_results.append({
            'original_idx':         idx,
            'ticker':               row['ticker'],
            'earnings_date':        row['earnings_date'],
            'positive_ratio':       np.nan,
            'negative_ratio':       np.nan,
            'neutral_ratio':        np.nan,
            'net_sentiment':        np.nan,
            'sentiment_volatility': np.nan,
            'n_chunks':             0
        })
    else:
        try:
            scores = score_transcript(text)
        except Exception as e:
            scores = {
                'positive_ratio':       np.nan,
                'negative_ratio':       np.nan,
                'neutral_ratio':        np.nan,
                'net_sentiment':        np.nan,
                'sentiment_volatility': np.nan,
                'n_chunks':             0
            }

        scores['original_idx']  = idx
        scores['ticker']        = row['ticker']
        scores['earnings_date'] = row['earnings_date']
        qa_results.append(scores)

    processed_qa += 1

    # Progress update
    if processed_qa % 100 == 0 and processed_qa > 0:
        elapsed        = time.time() - start_time_qa
        rate           = processed_qa / elapsed
        remaining_rows = len(df_merged) - len(qa_completed_idxs) - processed_qa
        remaining      = max(0, remaining_rows) / rate / 60
        print(f"  Q&A [{processed_qa}/{len(df_merged)}] | "
              f"Rate: {rate:.2f} rows/s | "
              f"ETA: {remaining:.1f}m")

    # ── Checkpoint block runs for all rows — both valid and NaN ───────────────
    if len(qa_results) % CHECKPOINT_EVERY == 0:
        batch_df          = pd.DataFrame(qa_results)
        qa_checkpoint_df  = pd.concat([qa_checkpoint_df, batch_df], ignore_index=True)
        qa_checkpoint_df.to_csv(QA_CHECKPOINT, index=False)
        qa_completed_idxs.update(batch_df['original_idx'].tolist())
        qa_results = []
        print(f"  ✓ Q&A checkpoint saved: {len(qa_checkpoint_df)} rows total")

# Save remaining
if qa_results:
    batch_df         = pd.DataFrame(qa_results)
    qa_checkpoint_df = pd.concat([qa_checkpoint_df, batch_df], ignore_index=True)
    qa_checkpoint_df.to_csv(QA_CHECKPOINT, index=False)
    qa_completed_idxs.update(batch_df['original_idx'].tolist())

print(f"\nQ&A scoring complete: {len(qa_checkpoint_df)} rows")
print(f"Saved: {QA_CHECKPOINT}")

Scoring Q&A section (analyst_text)...
No Q&A checkpoint found — starting from scratch
  Q&A [100/14738] | Rate: 4.68 rows/s | ETA: 52.1m
  Q&A [200/14738] | Rate: 4.52 rows/s | ETA: 53.6m
  Q&A [300/14738] | Rate: 4.23 rows/s | ETA: 56.9m
  Q&A [400/14738] | Rate: 4.25 rows/s | ETA: 56.2m
  Q&A [500/14738] | Rate: 4.35 rows/s | ETA: 54.6m
  ✓ Q&A checkpoint saved: 500 rows total
  Q&A [600/14738] | Rate: 4.26 rows/s | ETA: 53.4m
  Q&A [700/14738] | Rate: 4.19 rows/s | ETA: 53.8m
  Q&A [800/14738] | Rate: 4.26 rows/s | ETA: 52.5m
  Q&A [900/14738] | Rate: 4.28 rows/s | ETA: 52.0m
  Q&A [1000/14738] | Rate: 4.26 rows/s | ETA: 51.7m
  ✓ Q&A checkpoint saved: 1000 rows total
  Q&A [1100/14738] | Rate: 4.29 rows/s | ETA: 49.1m
  Q&A [1200/14738] | Rate: 4.29 rows/s | ETA: 48.7m
  Q&A [1300/14738] | Rate: 4.27 rows/s | ETA: 48.6m
  Q&A [1400/14738] | Rate: 4.24 rows/s | ETA: 48.5m
  Q&A [1500/14738] | Rate: 4.22 rows/s | ETA: 48.3m
  ✓ Q&A checkpoint saved: 1500 rows total
  Q&A [1600/14738]

In [9]:
# ── Merge mgmt and Q&A scores back into main dataframe ────────────────────────
print("Merging mgmt and Q&A scores into main dataframe...")

# Keep all columns from mgmt
mgmt_cols = mgmt_checkpoint_df.rename(columns={
    'positive_ratio':       'mgmt_positive_ratio',
    'negative_ratio':       'mgmt_negative_ratio',
    'neutral_ratio':        'mgmt_neutral_ratio',
    'net_sentiment':        'mgmt_sentiment',
    'sentiment_volatility': 'mgmt_sentiment_volatility',
    'n_chunks':             'mgmt_n_chunks'
})
mgmt_cols = mgmt_cols[[
    'original_idx',
    'mgmt_positive_ratio',
    'mgmt_negative_ratio',
    'mgmt_neutral_ratio',
    'mgmt_sentiment',
    'mgmt_sentiment_volatility',
    'mgmt_n_chunks'
]]

# Keep all columns from Q&A
qa_cols = qa_checkpoint_df.rename(columns={
    'positive_ratio':       'qa_positive_ratio',
    'negative_ratio':       'qa_negative_ratio',
    'neutral_ratio':        'qa_neutral_ratio',
    'net_sentiment':        'qa_sentiment',
    'sentiment_volatility': 'qa_sentiment_volatility',
    'n_chunks':             'qa_n_chunks'
})
qa_cols = qa_cols[[
    'original_idx',
    'qa_positive_ratio',
    'qa_negative_ratio',
    'qa_neutral_ratio',
    'qa_sentiment',
    'qa_sentiment_volatility',
    'qa_n_chunks'
]]

if 'original_idx' not in df_merged.columns:
    df_merged['original_idx'] = df_merged.index

df_merged = df_merged.merge(mgmt_cols, on='original_idx', how='left')
df_merged = df_merged.merge(qa_cols,  on='original_idx', how='left')
df_merged = df_merged.drop(columns=['original_idx'])

print(f"Columns after merge: {df_merged.columns.tolist()}")
print(f"Shape after merge:   {df_merged.shape}")

# ── Spot check: mgmt vs Q&A sentiment comparison ─────────────────────────────
print(f"\nSample output:")
print(df_merged[[
    'ticker', 'earnings_date',
    'net_sentiment', 'mgmt_sentiment', 'qa_sentiment',
    'mgmt_positive_ratio', 'qa_positive_ratio'
]].head(10).to_string())

print(f"\nMgmt vs Q&A sentiment comparison:")
print(f"  Mean net_sentiment:  {df_merged['net_sentiment'].mean():.4f}")
print(f"  Mean mgmt_sentiment: {df_merged['mgmt_sentiment'].mean():.4f}")
print(f"  Mean qa_sentiment:   {df_merged['qa_sentiment'].mean():.4f}")
print(f"\n  Corr (mgmt vs full):  {df_merged['net_sentiment'].corr(df_merged['mgmt_sentiment']):.4f}")
print(f"  Corr (qa vs full):    {df_merged['net_sentiment'].corr(df_merged['qa_sentiment']):.4f}")
print(f"  Corr (mgmt vs label): {df_merged['mgmt_sentiment'].corr(df_merged['label_5d']):.4f}")
print(f"  Corr (qa vs label):   {df_merged['qa_sentiment'].corr(df_merged['label_5d']):.4f}")

# ── Save intermediate merged file ─────────────────────────────────────────────
df_merged.to_csv('Outputs/sentiment_extraction/sentiment_merged_step6.csv', index=False)
print(f"\nSaved: sentiment_merged_step6.csv")

Merging mgmt and Q&A scores into main dataframe...
Columns after merge: ['ticker', 'earnings_date', 'q', 'earnings_date_is_approx', 'transcript', 'transcript_clean', 'speaker_parse_success', 'return_5d', 'return_10d', 'return_20d', 'label_5d', 'label_10d', 'label_20d', 'exec_text', 'analyst_text', 'positive_ratio', 'negative_ratio', 'neutral_ratio', 'net_sentiment', 'sentiment_volatility', 'n_chunks', 'mgmt_positive_ratio', 'mgmt_negative_ratio', 'mgmt_neutral_ratio', 'mgmt_sentiment', 'mgmt_sentiment_volatility', 'mgmt_n_chunks', 'qa_positive_ratio', 'qa_negative_ratio', 'qa_neutral_ratio', 'qa_sentiment', 'qa_sentiment_volatility', 'qa_n_chunks']
Shape after merge:   (14738, 33)

Sample output:
  ticker earnings_date  net_sentiment  mgmt_sentiment  qa_sentiment  mgmt_positive_ratio  qa_positive_ratio
0   BILI    2020-08-27       0.593890        0.676782      0.550691             0.688971           0.560581
1    GFF    2020-07-30       0.334645        0.620926      0.238361           